In [ ]:
pip install sentence-transformers faiss-cpu datasets

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import faiss
from typing import List, Dict, Tuple, Any
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import ndcg_score


### 1. Configuration
# All key parameters for the experiment are defined here.
# This makes it easy to tweak the model, dataset, or experiment size.

In [ ]:
# --- Configuration ---
MODEL_NAME = "mixedbread-ai/mxbai-embed-xsmall-v1"
DATASET_NAME = "mteb/hotpotqa"
CORPUS_SPLIT = "corpus"
QUERIES_SPLIT = "queries"
QRELS_SPLIT = "test"
TEXT_COLUMN_NAME = "text" # Column name in the dataset containing the text to embed

# --- Experiment Parameters ---
CORPUS_SIZE_TARGET = 100000 # The number of documents we aim to index
BATCH_SIZE = 10000
NUM_EVAL_QUERIES = 1000
DIMENSIONS = [384, 256, 128, 64]

# --- FAISS & Retrieval Parameters ---
HNSW_M = 32  # Number of connections per node in the HNSW graph
TOP_K = 10   # Number of results to retrieve for evaluation


### 2. Data Preparation
# This section loads the corpus, queries, and relevance data (qrels) from Hugging Face.
# It then prepares the data for the experiment by:
# 1. **Token Length Filtering**: Removing any documents from the corpus that exceed the model's maximum context window to ensure a fair evaluation.
# 2. Filtering the corpus to the desired size (`CORPUS_SIZE_TARGET`).
# 3. Creating a relevance map (`relevant_map`) that links queries to the original IDs of their correct documents.
# 4. Filtering the queries to ensure we only evaluate on queries that have at least one relevant document in our indexed corpus.

In [ ]:
def load_and_prepare_data() -> Tuple[Dataset, List[Dict], Dict[str, List[int]]]:
    # --- 1. Load Model and Tokenizer for Pre-processing ---
    print("Loading model to get tokenizer and max sequence length...")
    model_for_tokenizer = SentenceTransformer(MODEL_NAME)
    max_seq_length = model_for_tokenizer.max_seq_length
    print(f"Model max sequence length: {max_seq_length} tokens")

    # --- 2. Load Queries and Qrels ---
    print("Loading Queries, and Qrels...")
    queries_ds = load_dataset(DATASET_NAME, QUERIES_SPLIT, split=QUERIES_SPLIT)
    qrels_ds = load_dataset(DATASET_NAME, "default", split=QRELS_SPLIT)

    # --- 3. Efficiently Filter Corpus by Token Length ---
    print(f"Streaming corpus to find {CORPUS_SIZE_TARGET} documents with less than {max_seq_length} tokens...")
    streaming_corpus = load_dataset(DATASET_NAME, CORPUS_SPLIT, split=CORPUS_SPLIT, streaming=True)

    valid_docs = []
    for doc in streaming_corpus:
        if len(valid_docs) >= CORPUS_SIZE_TARGET:
            print(f"Found {len(valid_docs)} valid documents. Stopping stream.")
            break

        # Tokenize a single document to check its length
        num_tokens = len(model_for_tokenizer.tokenizer(doc[TEXT_COLUMN_NAME], truncation=False, padding=False)['input_ids'])

        if num_tokens < max_seq_length:
            # FIX: Enforce integer type for _id at the earliest point
            doc['_id'] = int(doc['_id'])
            valid_docs.append(doc)

    if len(valid_docs) < CORPUS_SIZE_TARGET:
        print(f"Warning: Found only {len(valid_docs)} documents meeting the criteria, which is less than the target of {CORPUS_SIZE_TARGET}.")

    # Convert the collected list of dictionaries into a Hugging Face Dataset
    corpus_subset = Dataset.from_list(valid_docs)
    print(f"Final corpus size for experiment: {len(corpus_subset)} documents.")

    # --- 4. Create Relevance Map and Filter Queries ---
    # indexed_corpus_ids is now a set of integers
    indexed_corpus_ids = set(corpus_subset["_id"])
    relevant_map = {}
    for rel in qrels_ds:
        q_id = rel["query-id"]
        # Cast corpus-id from string to int to match the type in indexed_corpus_ids
        c_id = int(rel["corpus-id"])

        if c_id in indexed_corpus_ids:
            relevant_map.setdefault(q_id, []).append(c_id)

    valid_query_ids = list(relevant_map.keys())
    eval_queries = [q for q in queries_ds if q["_id"] in valid_query_ids][:NUM_EVAL_QUERIES]
    print(f"num of eval_queries {len(eval_queries)}")
    print(f"Data preparation complete. Testing against {len(eval_queries)} queries.")

    return corpus_subset, eval_queries, relevant_map

# Execute Data Loading
corpus_subset, eval_queries, relevant_map = load_and_prepare_data()


### 3. Indexing & Embedding
# This section contains the core FAISS logic.
# - `instantiate_indexes`: Sets up three types of FAISS indexes for our experiment.
# - `embed`: A function to encode documents using the Sentence Transformer model and add them to the FAISS indexes.
# - `get_index_size`: A utility to measure the on-disk size of a FAISS index.

In [ ]:
def instantiate_indexes(dim: int) -> Tuple[faiss.Index, faiss.Index, faiss.Index]:
  # "HNSW32,SQ8" means an HNSW graph with 32 connections per node, using Scalar Quantization to 8-bit integers.
  scalar_q_factory_string = f"HNSW{HNSW_M},SQ8"

  # Create the base indexes
  base_nq_index = faiss.IndexHNSWFlat(dim, HNSW_M)
  base_scalar_q_index = faiss.index_factory(dim, scalar_q_factory_string)
  base_binary_q_index = faiss.IndexBinaryHNSW(dim, HNSW_M)

  # Wrap with IndexIDMap to allow using original integer IDs
  nq_index = faiss.IndexIDMap(base_nq_index)
  scalar_q_index = faiss.IndexIDMap(base_scalar_q_index)
  binary_q_index = faiss.IndexBinaryIDMap(base_binary_q_index)

  # Forward the training-related attributes from the base index to the IDMap wrapper
  # This is necessary because the SQ8 index needs to be trained.
  scalar_q_index.is_trained = base_scalar_q_index.is_trained
  scalar_q_index.train = base_scalar_q_index.train

  return nq_index, scalar_q_index, binary_q_index

def embed(dim: int, nq_index: faiss.Index, scalar_q_index: faiss.Index, binary_q_index: faiss.Index, corpus_dataset: Dataset):
  # Initialize the model with truncate_dim. This is the efficient, canonical way to use Matryoshka models.
  model = SentenceTransformer(MODEL_NAME, truncate_dim=dim)
  total_to_encode = len(corpus_dataset)
  is_trained = False

  for i in range(0, total_to_encode, BATCH_SIZE):
      # Get batch
      end_idx = min(i + BATCH_SIZE, total_to_encode)
      batch = corpus_dataset.select(range(i, end_idx))
      batch_texts = batch[TEXT_COLUMN_NAME]
      batch_ids = np.array(batch["_id"]).astype('int64') # FAISS requires int64 IDs

      # Encode vectors directly to the target dimension
      vectors = model.encode(batch_texts,
                             precision="float32",
                             convert_to_numpy=True,
                             normalize_embeddings=True).astype('float32')

      # Add to FAISS with original IDs
      nq_index.add_with_ids(vectors, batch_ids)
      if not is_trained:
        scalar_q_index.train(vectors)
        is_trained = True
      scalar_q_index.add_with_ids(vectors, batch_ids)

      # For binary index, pack bits from the truncated vectors
      packed_binary = np.packbits(vectors > 0, axis=-1)
      binary_q_index.add_with_ids(packed_binary, batch_ids)

      print(f"Embedding Progress: {end_idx}/{total_to_encode}")

def get_index_size(faiss_index: faiss.Index) -> float:
    temp_filename = "temp_size_check.index"
    try:
        # For IDMap indexes, we need to write the underlying index to get an accurate size
        if isinstance(faiss_index, (faiss.IndexBinaryIDMap, faiss.IndexIDMap)):
            underlying_index = faiss_index.index
            if isinstance(faiss_index, faiss.IndexBinaryIDMap):
                 faiss.write_index_binary(underlying_index, temp_filename)
            else:
                 faiss.write_index(underlying_index, temp_filename)
        else: # Handle non-IDMap indexes just in case
            faiss.write_index(faiss_index, temp_filename)

        size_in_bytes = os.path.getsize(temp_filename)
        size_in_mb = size_in_bytes / (1024**2)
        print(f"Index Size: {size_in_mb:.2f} MB")
        return size_in_mb
    finally:
        if os.path.exists(temp_filename):
            os.remove(temp_filename)


### 4. Experiment Execution & Evaluation
# This section defines the evaluation metrics and the main experiment loop.
# - `evaluate_retrieval`: Calculates Recall@K and MRR@K for a given index.
# - `execute_combined_experiment`: Iterates through each dimension, builds the indexes, and runs the evaluation.

In [ ]:
def get_recall_at_k(gold_ids: List[int], retrieved_ids: List[int]) -> float:
    """Calculates recall@k: the proportion of relevant documents found in the top k."""
    if not gold_ids:
        return 0.0

    hits = len(set(retrieved_ids) & set(gold_ids))
    return hits / len(gold_ids)

def get_mrr_at_k(gold_ids: List[int], retrieved_ids: List[int]) -> float:
    """Returns reciprocal rank of the first gold document found."""
    for i, idx in enumerate(retrieved_ids):
        if idx in gold_ids:
            return 1.0 / (i + 1)
    return 0.0

def evaluate_retrieval(index: faiss.Index, search_vectors: np.ndarray, eval_queries: List[Dict], relevant_map: Dict[str, List[int]], k: int) -> Tuple[float, float]:
    """Performs search and calculates average Recall and MRR."""
    _, I = index.search(search_vectors, k)
    recalls, mrrs = [], []

    for i, query in enumerate(eval_queries):
        gold_ids = relevant_map.get(query["_id"], [])
        retrieved_ids = I[i]
        recalls.append(get_recall_at_k(gold_ids, retrieved_ids))
        mrrs.append(get_mrr_at_k(gold_ids, retrieved_ids))

    return np.mean(recalls), np.mean(mrrs)

def execute_combined_experiment(dimensions: List[int], corpus_dataset: Dataset, queries: List[Dict], rel_map: Dict[str, List[int]]) -> pd.DataFrame:
    all_results = []

    for dim in dimensions:
        print(f"\n" + "="*50)
        print(f"RUNNING EXPERIMENT FOR DIMENSION: {dim}")
        print("="*50)

        # 1. Setup Indexes and Embed Corpus
        # nq = No Quantization, sq = Scalar Quantization, bq = Binary Quantization
        nq_idx, sq_idx, bq_idx = instantiate_indexes(dim)
        embed(dim, nq_idx, sq_idx, bq_idx, corpus_dataset)

        # 2. Embed Queries for this specific dimension
        query_texts = [f"{q[TEXT_COLUMN_NAME]}" for q in queries]
        model = SentenceTransformer(MODEL_NAME, truncate_dim=dim)

        print(f"\nEncoding {len(query_texts)} queries at dim={dim}...")
        q_vectors = model.encode(
            query_texts,
            precision="float32",
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype('float32')

        # 3. Size Metrics
        print("\nCalculating index sizes...")
        sizes = [get_index_size(nq_idx), get_index_size(sq_idx), get_index_size(bq_idx)]

        # 4. Retrieval Metrics
        print("\nCalculating retrieval accuracy...")
        r_nq, m_nq = evaluate_retrieval(nq_idx, q_vectors, queries, rel_map, k=TOP_K)
        r_sq, m_sq = evaluate_retrieval(sq_idx, q_vectors, queries, rel_map, k=TOP_K)

        q_bits = np.packbits(q_vectors > 0, axis=-1)
        r_bq, m_bq = evaluate_retrieval(bq_idx, q_bits, queries, rel_map, k=TOP_K)

        # 5. Append results
        all_results.append([dim, "No Quant", sizes[0], r_nq, m_nq])
        all_results.append([dim, "Scalar",   sizes[1], r_sq, m_sq])
        all_results.append([dim, "Binary",   sizes[2], r_bq, m_bq])

        print(f"\n--- Results for Dim={dim} ---")
        print(f"No Quant: Recall@{TOP_K}={r_nq:.4f}, MRR@{TOP_K}={m_nq:.4f}")
        print(f"Scalar:   Recall@{TOP_K}={r_sq:.4f}, MRR@{TOP_K}={m_sq:.4f}")
        print(f"Binary:   Recall@{TOP_K}={r_bq:.4f}, MRR@{TOP_K}={m_bq:.4f}")

    return pd.DataFrame(all_results, columns=["Dim", "Type", "Size_MB", f"Recall@{TOP_K}", f"MRR@{TOP_K}"])

# --- EXECUTE ---
results_df = execute_combined_experiment(DIMENSIONS, corpus_subset, eval_queries, relevant_map)

print("\n--- Final Performance Summary ---")
print(results_df.to_string(index=False))


### 5. Results & Visualization
# This section contains multiple plots to analyze the trade-offs between index size, retrieval accuracy (Recall/MRR), and vector dimensionality.

In [ ]:
# Set the style for all plots
sns.set_theme(style="whitegrid")

# --- Plot 1: Accuracy vs. Index Size ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Performance vs. Efficiency Trade-offs", fontsize=16, fontweight='bold')

sns.scatterplot(data=results_df, x="Size_MB", y=f"Recall@{TOP_K}", hue="Type", style="Dim", s=150, ax=ax1)
ax1.set_title(f"Recall@{TOP_K} vs. Index Size (MB)", fontsize=13)
ax1.set_xlabel("Index Size (MB) - Lower is better")
ax1.set_ylabel(f"Recall@{TOP_K} - Higher is better")

# --- Plot 2: Matryoshka Decay (Recall across Dimensions) ---
sns.lineplot(data=results_df, x="Dim", y=f"Recall@{TOP_K}", hue="Type", marker="o", ax=ax2)
ax2.set_title("Matryoshka Dimensionality Decay", fontsize=13)
ax2.set_xlabel("Vector Dimension")
ax2.set_ylabel(f"Recall@{TOP_K}")
ax2.invert_xaxis() # Show higher dims on the left

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
# --- Plot 3: Relative Performance Drop ---
# This plot shows how much accuracy is lost for each configuration compared to the best-performing one (384d, No Quant).

# 1. Map the 384d results as baselines for each category
baselines = results_df[results_df["Dim"] == 384].set_index("Type")[f"Recall@{TOP_K}"].to_dict()

# 2. Calculate Drop relative to the 384d version of the SAME type
def calculate_relative_drop(row):
    base_val = baselines[row["Type"]]
    return ((row[f"Recall@{TOP_K}"] - base_val) / base_val) * 100

results_df["Recall_Drop_Relative_Pct"] = results_df.apply(calculate_relative_drop, axis=1)

# 3. Setup Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Relative Performance Drop Analysis", fontsize=16, fontweight='bold')

# --- Plot 3a: Efficiency Frontier (Size vs Relative Drop) ---
sns.scatterplot(
    data=results_df, x="Size_MB", y="Recall_Drop_Relative_Pct",
    hue="Type", style="Dim", s=150, ax=ax1, palette="viridis"
)
ax1.set_title("Relative Recall Drop (%) vs. Size\n(Compared to 384d of same Type)", fontsize=13)
ax1.set_xlabel("Index Size (MB)")
ax1.set_ylabel("Relative Recall Drop (%)")
ax1.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.axhline(0, color='black', linestyle='--', alpha=0.5)

# --- Plot 3b: Pure Matryoshka Decay ---
sns.lineplot(
    data=results_df, x="Dim", y="Recall_Drop_Relative_Pct",
    hue="Type", marker="o", linewidth=3, markersize=10, ax=ax2, palette="viridis"
)
ax2.set_title("Matryoshka Dimensionality Decay\n(Retention within Category)", fontsize=13)
ax2.set_xlabel("Vector Dimension")
ax2.set_ylabel("Relative Recall Drop (%)")
ax2.invert_xaxis()
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.axhline(0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
# --- Plot 4: Storage Savings ---
# This bar chart clearly shows the percentage of storage saved for each configuration compared to the largest, uncompressed index.

# 1. Dynamically find the baseline size (Max Dimension, No Quantization)
max_dim = results_df['Dim'].max()
baseline_size = results_df[(results_df['Dim'] == max_dim) & (results_df['Type'] == 'No Quant')]['Size_MB'].values[0]

# 2. Calculate % Reduction vs. Baseline
results_df["Space_Reduction_Pct"] = ((1 - (results_df["Size_MB"] / baseline_size)) * 100).clip(lower=0)

# 3. Plot Configuration
plt.figure(figsize=(12, 7))
ax = sns.barplot(
    data=results_df,
    x="Dim",
    y="Space_Reduction_Pct",
    hue="Type",
    palette="viridis",
    order=sorted(results_df['Dim'].unique(), reverse=True)
)

# 4. Styling for a publication-ready look
plt.title(f"Total Space Reduction (%) vs. {max_dim}-Dim Baseline (Float32)", fontsize=15, fontweight='bold', pad=15)
plt.ylabel("Storage Saved (%)", fontsize=12)
plt.xlabel("Embedding Dimension (MRL)", fontsize=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())

# Add percentage labels on top of the bars
for p in ax.patches:
    height = p.get_height()
    if height > 0.5:
        ax.annotate(f'{height:.1f}%',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='center',
                    xytext=(0, 9),
                    textcoords='offset points',
                    fontsize=10)

plt.tight_layout()
plt.savefig('storage_savings_bar_chart.png', dpi=300, bbox_inches='tight')
print("Bar chart successfully saved as 'storage_savings_bar_chart.png'")
plt.show()
